# Unit 6 — Python SQL Database Access
**Course:** BT151CO — Object-Oriented Programming  
**Unit:** 6 of 8  
**Duration:** 5 Hours  

---

## What Is This Notebook?

This notebook teaches you how to store, retrieve, and manage data using **SQLite** — a database engine that is built into Python. Every code cell uses an **in-memory database** (`':memory:'`) or a **temporary file** that is cleaned up automatically, so you can re-run cells as many times as you like.

> **No installation required** — `sqlite3` is part of the Python standard library.

---

## Learning Objectives

By the end of this unit you will be able to:

1. Explain what a relational database is and why programs use one
2. Connect to a SQLite database file from Python
3. Create tables with appropriate column types and constraints
4. Insert, read, update, and delete records using **parameterised** SQL queries
5. Explain what a **transaction** is and control it with `commit()` and `rollback()`
6. Handle database errors gracefully with `try / except`
7. Build a small data-driven application that **persists data between runs**

## Table of Contents

1. [Introduction to Databases & SQL](#s1)
2. [Connecting to a Database](#s2)
3. [Creating Tables](#s3)
4. [INSERT — Adding Data](#s4)
5. [READ (SELECT) — Querying Data](#s5)
6. [UPDATE & DELETE — Modifying Data](#s6)
7. [COMMIT & ROLLBACK — Transactions](#s7)
8. [Error Handling](#s8)
9. [Summary & Checklist](#summary)

<a id="s1"></a>
---
## Section 1 — Introduction to Databases & SQL

### Why Not Just Use Files?

When your program closes, variables disappear. Files can save data — but they have serious limitations:

| Approach | Problem |
|---|---|
| Plain text / CSV | Hard to search; no types; full rewrite to change one value |
| `pickle` / JSON | Full file in memory; no query language; fragile |
| **Database** | Structured, queryable, efficient, persistent, concurrent-safe |

> **Analogy:** A database is a filing cabinet in a school office — each drawer is a table, each folder is a row, each piece of paper is a column. Finding any student in seconds beats searching a pile of loose papers.



### What Is a Relational Database?

Data is organised into **tables**:

- Each **column** has a name and a data type
- Each **row** is one record (one student, one order, one product)
- The **primary key** (`id`) uniquely identifies every row
- Tables can be **linked** to one another using **relationships**, using 
    - **primary keys** (column (or set of columns) that uniquely identifies each record in a table) and 
    - **foreign keys** (column (or set of columns) that links one table to another by referencing the primary key of another table).

![primary-foreign-key](https://thecrazyprogrammer.com/wp-content/uploads/2019/04/Difference-between-Primary-Key-and-Foreign-Key-1024x672.gif)

Relational vs Non-Relational Database: 

| Feature            | Relational Database (SQL)                          | Non-Relational Database (NoSQL)                                                                       |
| ------------------ | -------------------------------------------------- | ----------------------------------------------------------------------------------------------------- |
| **Data structure** | Tables with rows and columns                       | Documents, key-value pairs, graphs, or wide-column stores                                             |
| **Schema**         | Fixed, predefined schema                           | Flexible or schema-less                                                                               |
| **Relationships**  | Built-in using keys and joins                      | Usually stores related data together instead of using joins                                           |
| **Query language** | SQL (Structured Query Language)                    | Varies by database; not always SQL                                                                    |
| **Scalability**    | Typically scales vertically (more powerful server) | Often designed to scale horizontally (more servers)                                                   |
| **Consistency**    | Strong consistency and ACID transactions           | Often optimized for scalability and availability, though many support transactions to varying degrees |
| **Best for**       | Structured, consistent data                        | Large volumes of changing or unstructured data                                                        |

**Use a non-relational database when:**
- Your data structure changes frequently.
- You need to handle very large amounts of data across many servers.
- You're building applications with rapidly evolving requirements, such as social media platforms, IoT systems, or real-time analytics.

### What Is SQL?

**Structured Query Language** — five commands cover 95% of all work:

```sql
-- Create a table
CREATE TABLE students (id INTEGER, name TEXT, gpa REAL);

-- Add a record
INSERT INTO students VALUES (1, 'Alice', 3.8);

-- Read records
SELECT name, gpa FROM students WHERE gpa > 3.0;

-- Update a record
UPDATE students SET gpa = 3.9 WHERE id = 1;

-- Delete a record
DELETE FROM students WHERE id = 1;
```

SQL reads almost like English — CREATE the shape, INSERT data, SELECT to read, UPDATE to change, DELETE to remove.

**Note**: 

- Teaching SQL is not a part of the syllabus. Please refer to [https://sqlcrashcourse.com](https://sqlcrashcourse.com) for a crash course in SQL. 
- Or, refer to the [sql-cheet-sheet](https://www.geeksforgeeks.org/sql/sql-cheat-sheet/) by GeeksforGeeks. 

### What Is SQLite?

SQLite is a database engine with **no server required** — the entire database lives in a single `.db` file on disk. It is built into Python:

```python
import sqlite3   # already in Python — no pip install
```

SQLite is the most widely deployed database engine in the world — used in Android, iOS, Firefox, Chrome, and Dropbox.

<a id="s2"></a>
---
## Section 2 — Connecting to a Database

### The Four-Step Pattern

Every database interaction generally follows these steps:

1. **Connect** — `sqlite3.connect(filename)` creates the database file if it doesn't exist.
2. **Create a cursor** — `conn.cursor()` creates the object that executes SQL statements.
3. **Execute SQL** — `cursor.execute(sql, params)`.
4. **Fetch results (if the query returns rows)**:
   - `cursor.fetchone()`
   - `cursor.fetchmany(n)`
   - `cursor.fetchall()`

> **Analogy:** A connection is your library card (authorisation). A cursor is the librarian who carries out your requests.


No installation is required. Any Python 3.x installation includes it.

In [1]:
import sqlite3
print(sqlite3.version)        # sqlite3 module version
print(sqlite3.sqlite_version)  # underlying SQLite library version

2.6.0
3.49.1


C:\Users\HP\AppData\Local\Temp\ipykernel_31612\3001796262.py:2: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(sqlite3.version)        # sqlite3 module version


### The Context Manager — `with` Statement

Always prefer the `with` statement. It:
- **Commits** automatically when the block exits without error
- **Rolls back** automatically if an exception is raised
- **Closes** the connection when the block ends

```python
with sqlite3.connect('school.db') as conn:
    cursor = conn.cursor()
    cursor.execute('...')
# connection closed automatically here
```

> ⚠️ **Best Practice:** Always use the `with` statement. Never leave a connection open.

### Using DB Browser for SQLite

When learning, it helps to **see** inside your database file as well as query it.
**DB Browser for SQLite** (https://sqlitebrowser.org/) is a free, open-source GUI tool
that lets you open `.db` files, browse tables, and run SQL manually.

**Recommended learning workflow:**

1. Write Python code that creates/modifies the database.
2. Open the `.db` file in DB Browser to visually inspect the result.
3. This builds intuition about what your code is actually doing.

You do **not** need DB Browser to follow this unit — it is optional but helpful.

In [9]:
# ── Connecting to an in-memory database ──────────────────────────────────
# ':memory:' creates a temporary database that lives only while this
# Python session is running. Perfect for learning — nothing is written to disk.

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute('SELECT sqlite_version()')
    version = cursor.fetchone()
    print(f'SQLite version: {version}')

print('Connection closed automatically by the with statement.')

SQLite version: ('3.49.1',)
Connection closed automatically by the with statement.


<a id="s3"></a>
---
## Section 3 — Creating Tables

### SQLite Data Types

| SQLite Type | Python type | Use for |
|---|---|---|
| `INTEGER` | `int` | IDs, counts, flags, booleans |
| `REAL` | `float` | Prices, GPAs, scores |
| `TEXT` | `str` | Names, descriptions, dates |
| `BLOB` | `bytes` | Images, binary files |
| `NULL` | `None` | Missing / unknown values |

> **Date tip:** Store dates as `TEXT` in `'YYYY-MM-DD'` format — it sorts correctly as a string and is human-readable.


**TEXT** vs **VARCHAR(n)**
> **SQLite note:** `TEXT` is usually preferred over `VARCHAR(n)`. SQLite does **not** enforce the length specified by `VARCHAR(n)`, so `TEXT` and `VARCHAR(n)` behave almost the same. If you need a maximum length, add a `CHECK` constraint, for example:
>
> ```sql
> CHECK (length(owner_name) <= 50)
> ```

Please check [MySQL-vs-SQLite.md](../Notes/MySQL-vs-SQLite.md) for more information. 

### Constraints

| Constraint | Effect |
|---|---|
| `PRIMARY KEY` | Unique identifier — no duplicates |
| `NOT NULL` | Column must always have a value |
| `UNIQUE` | No two rows can share this value |
| `DEFAULT value` | Used when INSERT omits the column |
| `CHECK (condition)` | Row rejected if condition is false |



#### The `CREATE TABLE` statement

A **table** defines the structure of the data you want to store.
Creating a table is like designing a form: you decide how many fields there are
and what type of data each field holds.

```sql
CREATE TABLE students (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    name    TEXT    NOT NULL,
    gpa     REAL    DEFAULT 0.0,
    enrolled DATE
);
```

In Python:

In [2]:
import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE student (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            name     TEXT    NOT NULL,
            gpa      REAL    DEFAULT 0.0,
            enrolled TEXT
        )
    """
    )

    # Verify the table was created
    cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )
    tables = cursor.fetchall()
    print('Tables in database:', tables)

Tables in database: [('student',), ('sqlite_sequence',)]


### The `IF NOT EXISTS` Clause

Always include `IF NOT EXISTS` — it makes table creation **idempotent** (safe to run every time your app starts, even if the table already exists).

In [3]:
# ── CREATE TABLE — students ───────────────────────────────────────────────

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            name     TEXT    NOT NULL,
            gpa      REAL    DEFAULT 0.0,
            enrolled TEXT
        )
    """)

    # Verify the table was created
    cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )
    
    tables = cursor.fetchall()
    print('Tables in database:', tables)

Tables in database: [('students',), ('sqlite_sequence',)]


In [6]:
# ── CREATE TABLE 
# — bank_accounts with CHECK constraint 

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS bank_accounts (
            account_id   INTEGER  PRIMARY KEY AUTOINCREMENT,
            owner_name   TEXT     NOT NULL,
            balance      REAL     NOT NULL  DEFAULT 0.0,
            account_type TEXT     NOT NULL  DEFAULT 'current',
            CHECK (balance >= 0)   -- balance can NEVER go negative
        )
    """)

    print('bank_accounts table created with CHECK constraint.')

    # Try to insert a negative balance — the CHECK will reject it
    try:
        cursor.execute(
            "INSERT INTO bank_accounts (owner_name, balance) VALUES ('Alice', -50.0)"
            # "INSERT INTO bank_accounts (owner_name, balance) VALUES ('Alice', 50.0)"
        )
        print(f'Execution successful')
    except sqlite3.IntegrityError as e:
        print(f'Rejected by CHECK constraint\nError: {e}')
        

bank_accounts table created with CHECK constraint.
Rejected by CHECK constraint
Error: CHECK constraint failed: balance >= 0


<a id="s4"></a>
---
## Section 4 — INSERT — Adding Data

### Parameterised Queries — Non-Negotiable

Always use `?` placeholders — **never** concatenate user input into SQL:

```python
# ❌ DANGEROUS — SQL injection attack!
cursor.execute("SELECT * FROM students WHERE name = '" + name + "'")

# ✅ SAFE — parameterised query
cursor.execute('SELECT * FROM students WHERE name = ?', (name,))
```

> ⚠️ **SQL Injection** is OWASP Top 10 #3. It has caused some of the largest data breaches in history. Always use `?` placeholders — no exceptions, ever.

### The Trailing-Comma Trap

Single-element tuples **need a trailing comma**:

```python
# ❌ Wrong — ('Alice') is just 'Alice' in parentheses
cursor.execute('... WHERE name = ?', ('Alice'))   # TypeError!

# ✅ Correct — trailing comma makes it a 1-element tuple
cursor.execute('... WHERE name = ?', ('Alice',))
```

### `cursor.lastrowid`

After an INSERT, `cursor.lastrowid` gives you the `id` that was assigned by `AUTOINCREMENT`. Useful when you immediately need to reference the new row.

### `executemany()`

Insert a list of rows in one call — more efficient than a loop of `execute()` calls.

In [7]:
# ── INSERT a single row ──────────────────────────────────────────────────

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()

    # Create the table first
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            gpa REAL DEFAULT 0.0,
            enrolled TEXT
        )
    """)

    # Insert one row using parameterised query
    cursor.execute(
        'INSERT INTO students (name, gpa, enrolled) VALUES (?, ?, ?)',
        ('Alice', 3.8, '2024-09-01')   # always a tuple or list
    )
    conn.commit()

    print(f'Row inserted. New id: {cursor.lastrowid}')

    # Verify
    cursor.execute('SELECT * FROM students')
    print(cursor.fetchall())

Row inserted. New id: 1
[(1, 'Alice', 3.8, '2024-09-01')]


In [8]:
# ── executemany() — insert multiple rows efficiently ────────────────────

import sqlite3

new_students = [
    ('Bob',     2.9, '2024-09-01'),
    ('Charlie', 3.5, '2024-09-01'),
    ('Diana',   3.7, '2024-09-01'),
    ('Eve',     3.1, '2024-09-01'),
]

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            gpa REAL DEFAULT 0.0,
            enrolled TEXT
        )
    """)

    cursor.executemany(
        'INSERT INTO students (name, gpa, enrolled) VALUES (?, ?, ?)',
        new_students
    )
    conn.commit()

    print(f'Inserted {cursor.rowcount} rows.')
    cursor.execute('SELECT * FROM students')
    for row in cursor.fetchall():
        print(row)

Inserted 4 rows.
(1, 'Bob', 2.9, '2024-09-01')
(2, 'Charlie', 3.5, '2024-09-01')
(3, 'Diana', 3.7, '2024-09-01')
(4, 'Eve', 3.1, '2024-09-01')


<a id="s5"></a>
---
## Section 5 — READ (SELECT) — Querying Data

### Fetch Methods

| Method | Returns | Use when |
|---|---|---|
| `fetchone()` | One row or `None` | Expect exactly one result |
| `fetchmany(n)` | Up to n rows | Processing in batches |
| `fetchall()` | All rows | Small result sets |
| `for row in cursor` | One row at a time | Large tables |

Each row is a **Python tuple** — the position matches the column order in SELECT.

### The `WHERE` Clause

Filter rows that match a condition:

| Operator | Example |
|---|---|
| `=`, `!=` | `WHERE name = 'Alice'` |
| `>`, `<`, `>=`, `<=` | `WHERE gpa >= 3.5` |
| `BETWEEN` | `WHERE gpa BETWEEN 3.0 AND 3.5` |
| `LIKE` | `WHERE name LIKE 'A%'` |
| `IN` | `WHERE id IN (1, 3, 5)` |

### Sorting and Limiting

```sql
SELECT name, gpa FROM students ORDER BY gpa DESC LIMIT 5;
```

### Aggregate Functions

| Function | Purpose |
|---|---|
| `COUNT(*)` | Number of rows |
| `SUM(col)` | Total |
| `AVG(col)` | Mean |
| `MAX(col)` / `MIN(col)` | Largest / smallest |

In [9]:
# ── Helper: create and populate a students database ─────────────────────
# We define this once and reuse it throughout Section 5.

import sqlite3

def make_students_db():
    """Returns an in-memory connection with a populated students table."""
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            gpa  REAL DEFAULT 0.0,
            enrolled TEXT
        )
    """)
    cursor.executemany(
        'INSERT INTO students (name, gpa, enrolled) VALUES (?, ?, ?)',
        [
            ('Alice',   3.8, '2024-09-01'),
            ('Bob',     2.9, '2024-09-01'),
            ('Charlie', 3.5, '2024-09-01'),
            ('Diana',   3.7, '2024-09-01'),
            ('Eve',     3.1, '2024-09-01'),
        ]
    )
    conn.commit()
    return conn

print('make_students_db() ready to use.')

make_students_db() ready to use.


In [10]:
# ── SELECT — fetchall() ──────────────────────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

cursor.execute('SELECT id, name, gpa FROM students')
rows = cursor.fetchall()

print('All students:')
for row in rows:
    print(f'  {row}')

conn.close()

All students:
  (1, 'Alice', 3.8)
  (2, 'Bob', 2.9)
  (3, 'Charlie', 3.5)
  (4, 'Diana', 3.7)
  (5, 'Eve', 3.1)


In [11]:
# ── SELECT with WHERE — filter by GPA ───────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

# Students with GPA above 3.5
cursor.execute(
    'SELECT name, gpa FROM students WHERE gpa > ?',
    (3.5,)   # trailing comma — single-element tuple!
)

print('Students with GPA > 3.5:')
for name, gpa in cursor.fetchall():
    print(f'  {name}: {gpa}')

# Students whose name starts with a specific letter
cursor.execute(
    'SELECT name, gpa FROM students WHERE name LIKE ?',
    ('D%',)
)
print('Students whose name starts with D:')
print(cursor.fetchall())

conn.close()

Students with GPA > 3.5:
  Alice: 3.8
  Diana: 3.7
Students whose name starts with D:
[('Diana', 3.7)]


In [12]:
# ── ORDER BY and LIMIT ───────────────────────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

# Top 3 students by GPA
cursor.execute("""
    SELECT name, gpa
    FROM students
    ORDER BY gpa DESC
    LIMIT 3
""")

print('Top 3 students:')
for i, (name, gpa) in enumerate(cursor.fetchall(), start=1):
    print(f'  {i}. {name} — {gpa}')

conn.close()

Top 3 students:
  1. Alice — 3.8
  2. Diana — 3.7
  3. Charlie — 3.5


In [13]:
# ── Aggregate functions ───────────────────────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

cursor.execute('SELECT COUNT(*) FROM students')
total = cursor.fetchone()[0]

cursor.execute('SELECT AVG(gpa) FROM students')
avg_gpa = cursor.fetchone()[0]

cursor.execute('SELECT MAX(gpa), MIN(gpa) FROM students')
top, bottom = cursor.fetchone()

cursor.execute('SELECT SUM(gpa) FROM students')
total_gpa = cursor.fetchone()[0]

print(f'Total students : {total}')
print(f'Average GPA    : {avg_gpa:.2f}')
print(f'Highest GPA    : {top}')
print(f'Lowest GPA     : {bottom}')
print(f'Sum of all GPAs: {total_gpa:.2f}')

conn.close()

Total students : 5
Average GPA    : 3.40
Highest GPA    : 3.8
Lowest GPA     : 2.9
Sum of all GPAs: 17.00


<a id="s6"></a>
---
## Section 6 — UPDATE & DELETE — Modifying Data

### The Golden Rule

> ⚠️ **Always include a `WHERE` clause in `UPDATE` and `DELETE`.  
> Before running either, test your `WHERE` clause with a `SELECT` first.**

### UPDATE

```sql
UPDATE students SET gpa = 3.95 WHERE id = 1;
```

- `SET` lists which columns change and their new values
- `cursor.rowcount` tells you how many rows were changed
- If `rowcount` is 0, the `WHERE` clause matched nothing

### DELETE

| Statement | Effect |
|---|---|
| `DELETE FROM students WHERE id = 6` | Remove one specific row |
| `DELETE FROM students` | Remove ALL rows (keep table structure) |
| `DROP TABLE students` | Remove table entirely |

> `DROP TABLE` cannot be undone (unless inside a transaction that is rolled back).

In [ ]:
# ── UPDATE — modify one row ───────────────────────────────────────────────

import sqlite3

conn = make_students_db()
cursor = conn.cursor()

# Step 1: Verify with SELECT before updating
cursor.execute('SELECT id, name, gpa FROM students WHERE id = ?', 
               (1,)
               )
print('Before update:', cursor.fetchone())

# Step 2: Only then UPDATE
cursor.execute(
    'UPDATE students SET gpa = ? WHERE id = ?',
    (3.95, 1)
)
conn.commit()
print(f'Rows updated: {cursor.rowcount}')

# Confirm
cursor.execute('SELECT id, name, gpa FROM students WHERE id = ?', (1,))
print('After update: ', cursor.fetchone())

conn.close()

Before update: (1, 'Alice', 3.8)
Rows updated: 1
After update:  (1, 'Alice', 3.95)


In [19]:
# ── UPDATE — multiple columns at once ────────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

print(f'Before Update')
cursor.execute('SELECT * FROM students WHERE name = ?', ('Diana',))
print(cursor.fetchone())

cursor.execute(
    'UPDATE students SET gpa = ?, enrolled = ? WHERE name = ?',
    (4.0, '2024-01-15', 'Diana')
)
conn.commit()

print(f'\nAfter Update')
cursor.execute('SELECT * FROM students WHERE name = ?', ('Diana',))
print(cursor.fetchone())

conn.close()

Before Update
(4, 'Diana', 3.7, '2024-09-01')

After Update
(4, 'Diana', 4.0, '2024-01-15')


In [20]:
# ── DELETE — remove a specific row ───────────────────────────────────────

conn = make_students_db()
cursor = conn.cursor()

print('Before delete:')
cursor.execute('SELECT id, name FROM students')
print(cursor.fetchall())

# Remove Eve (id=5)
cursor.execute('DELETE FROM students WHERE name = ?', ('Eve',))
conn.commit()
print(f'Deleted {cursor.rowcount} row(s).')

print('After delete:')
cursor.execute('SELECT id, name FROM students')
print(cursor.fetchall())

conn.close()

Before delete:
[(1, 'Alice'), (2, 'Bob'), (3, 'Charlie'), (4, 'Diana'), (5, 'Eve')]
Deleted 1 row(s).
After delete:
[(1, 'Alice'), (2, 'Bob'), (3, 'Charlie'), (4, 'Diana')]


<a id="s7"></a>
---
## Section 7 — COMMIT & ROLLBACK — Transactions

### What Is a Transaction?

A **transaction** is a group of SQL operations that all succeed — or all fail.

**Bank transfer example:**
```
BEGIN TRANSACTION
    1. Deduct £100 from Alice's account   ← operation 1
    2. Add    £100 to Bob's account       ← operation 2
COMMIT   ← both OK: save permanently
```

If step 2 fails:
- **Without transaction:** Alice loses £100, Bob gets nothing
- **With transaction:** `ROLLBACK` — both operations are undone

### ACID Properties

| Letter | Property | Meaning |
|---|---|---|
| **A** | Atomic | All-or-nothing — no partial state |
| **C** | Consistent | Database rules are never violated |
| **I** | Isolated | Concurrent transactions don't interfere |
| **D** | Durable | Committed changes survive crashes |

### `commit()` vs `rollback()`

- `conn.commit()` — save all uncommitted changes permanently
- `conn.rollback()` — undo all uncommitted changes
- `with` statement — commits automatically on success, rolls back on error

> ⚠️ Changes are **not permanent** until `commit()` is called. If the program crashes before commit, all uncommitted changes are lost.

In [21]:
# ── commit() demo — changes exist in buffer until commit ─────────────────

import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute(
    'CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT, gpa REAL)'
)
conn.commit()

# Insert WITHOUT commit
cursor.execute(
    'INSERT INTO students VALUES (?, ?, ?)',
    (1, 'Alice', 3.8)
)
# Alice is in the buffer — we can read her from THIS connection
cursor.execute('SELECT * FROM students')
print('After INSERT, before commit:', cursor.fetchall())

# Now commit — row is permanently saved
conn.commit()
print('After commit: changes are permanent.')

conn.close()

After INSERT, before commit: [(1, 'Alice', 3.8)]
After commit: changes are permanent.


In [22]:
# ── rollback() demo — bank transfer simulation ────────────────────────────

import sqlite3

def setup_bank():
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE bank_accounts (
            account_id INTEGER PRIMARY KEY,
            owner_name TEXT NOT NULL,
            balance    REAL NOT NULL DEFAULT 0.0,
            CHECK (balance >= 0)
        )
    """)
    cursor.executemany(
        'INSERT INTO bank_accounts VALUES (?, ?, ?)',
        [(1, 'Alice', 500.0), (2, 'Bob', 200.0)]
    )
    conn.commit()
    return conn

def show_balances(conn):
    cursor = conn.cursor()
    cursor.execute('SELECT owner_name, balance FROM bank_accounts')
    for name, bal in cursor.fetchall():
        print(f'  {name}: £{bal:.2f}')

def transfer(conn, from_id, to_id, amount):
    cursor = conn.cursor()
    try:
        cursor.execute(
            'UPDATE bank_accounts SET balance = balance - ? WHERE account_id = ?',
            (amount, from_id)
        )
        cursor.execute(
            'UPDATE bank_accounts SET balance = balance + ? WHERE account_id = ?',
            (amount, to_id)
        )
        conn.commit()
        print(f'Transferred £{amount:.2f} successfully.')
    except sqlite3.Error as e:
        conn.rollback()
        print(f'Transfer failed: {e}. All changes rolled back.')

conn = setup_bank()
print('Starting balances:')
show_balances(conn)

print('\nTransferring £150 from Alice to Bob:')
transfer(conn, from_id=1, to_id=2, amount=150.0)
show_balances(conn)

print('\nTrying to transfer £1000 (Alice has only £350):')
transfer(conn, from_id=1, to_id=2, amount=1000.0)
show_balances(conn)

conn.close()

Starting balances:
  Alice: £500.00
  Bob: £200.00

Transferring £150 from Alice to Bob:
Transferred £150.00 successfully.
  Alice: £350.00
  Bob: £350.00

Trying to transfer £1000 (Alice has only £350):
Transfer failed: CHECK constraint failed: balance >= 0. All changes rolled back.
  Alice: £350.00
  Bob: £350.00


<a id="s8"></a>
---
## Section 8 — Handling Database Errors

### The `sqlite3` Exception Hierarchy

```
sqlite3.Error  (base class — catches everything)
├── sqlite3.DatabaseError
│   ├── sqlite3.IntegrityError   ← constraint violated
│   │                              (UNIQUE, NOT NULL, CHECK)
│   ├── sqlite3.OperationalError ← table not found,
│   │                              file locked, syntax error
│   └── sqlite3.DataError        ← invalid data type
└── sqlite3.InterfaceError       ← API misuse
```

### When to Catch Which Exception

| Exception | Triggered by |
|---|---|
| `IntegrityError` | Duplicate email, NULL in NOT NULL column, CHECK violated |
| `OperationalError` | Table doesn't exist, file permissions, SQL syntax error |
| `sqlite3.Error` | Catch-all for any unexpected database error |

### Pattern: Return a Result Flag

A clean pattern is to return `True`/`False` from database functions — the caller gets a clear success/failure signal without seeing database internals.

In [ ]:
# ── IntegrityError — UNIQUE constraint violation ─────────────────────────

import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE members (
        member_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name      TEXT NOT NULL,
        email     TEXT UNIQUE NOT NULL,
        joined    TEXT NOT NULL
    )
""")
conn.commit()

def add_member(conn, name, email, joined):
    try:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO members (name, email, joined) VALUES (?, ?, ?)',
            (name, email, joined)
        )
        conn.commit()
        print(f'  Registered: {name} ({email})')
        return True
    except sqlite3.IntegrityError:
        print(f'  Error: "{email}" is already registered.')
        return False
    except sqlite3.OperationalError as e:
        print(f'  Database error: {e}')
        return False

print('Registering members:')
add_member(conn, 'Alice',  'alice@example.com', '2024-09-01')  # OK
add_member(conn, 'Bob',    'bob@example.com',   '2024-09-01')  # OK
add_member(conn, 'Alicia', 'alice@example.com', '2024-09-01')  # Duplicate!

conn.close()

In [ ]:
# ── OperationalError — querying a table that doesn't exist ───────────────

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    try:
        cursor.execute('SELECT * FROM nonexistent_table')
    except sqlite3.OperationalError as e:
        print(f'OperationalError caught: {e}')

# Custom exception pattern — hide database details from the caller
class DuplicateEmailError(Exception):
    """Raised when a member email already exists."""

print('Custom exception class defined: DuplicateEmailError')

<a id="summary"></a>
---
## Summary & Checklist

### The CRUD Pattern

| Letter | SQL | Python |
|---|---|---|
| **C**reate | `INSERT INTO` | `cursor.execute(sql, params)` |
| **R**ead | `SELECT` | `cursor.fetchone()` / `fetchall()` |
| **U**pdate | `UPDATE … SET` | `cursor.execute(sql, params)` |
| **D**elete | `DELETE FROM` | `cursor.execute(sql, params)` |

### The Three Rules of Safe Database Programming

1. **Always PARAMETERISE** — use `?` placeholders, never string concatenation
2. **Always COMMIT** — or use `with` which does it automatically
3. **Always WHERE** — on every `UPDATE` and `DELETE`

### Common Mistakes Quick Reference

| # | Mistake | Fix |
|---|---|---|
| 1 | String concat in SQL | Always use `?` placeholders |
| 2 | Forgetting `commit()` | Use `with` or explicit commit |
| 3 | No `WHERE` on UPDATE/DELETE | Always target a specific row |
| 4 | Missing trailing comma `('Alice')` | `('Alice',)` — comma makes the tuple |
| 5 | `fetchall()` on a huge table | Iterate cursor directly |
| 6 | Not closing the connection | Use `with` context manager |
| 7 | Trusting `rowcount` on SELECT | Use `COUNT(*)` instead |

### Self-Check Checklist

Mark each box when you can do it without looking at notes:

- [ ] Connect to an in-memory database and check the SQLite version
- [ ] Create a table with `PRIMARY KEY`, `NOT NULL`, `UNIQUE`, and `CHECK` constraints
- [ ] Insert a single row using a parameterised query
- [ ] Insert multiple rows using `executemany()`
- [ ] Read all rows with `SELECT` and `fetchall()`
- [ ] Filter rows with `WHERE`, sort with `ORDER BY`, limit with `LIMIT`
- [ ] Use `COUNT(*)`, `AVG()`, `MAX()`, `MIN()` aggregate functions
- [ ] Update a specific row with `UPDATE … WHERE` and verify with `rowcount`
- [ ] Delete a specific row with `DELETE … WHERE`
- [ ] Wrap two operations in a transaction with `commit()` / `rollback()`
- [ ] Catch `IntegrityError` and `OperationalError` with `try / except`
- [ ] Explain SQL injection and why `?` placeholders prevent it

---

**Next Unit:** Unit 7 — Network Programming  
*Clients · Servers · Building network applications in Python*

---
*BT151CO — Object-Oriented Programming · Unit 6*